In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
metadata_df = spark.table("workspace.metadata.metadata_config") \
                   .filter(col("active_flag") == "Y")

display(metadata_df)

table_name,source_table,target_table,primary_key,watermark_column,load_type,active_flag,scd_type
users,workspace.raw.users,workspace.bronze.users,user_id,created_at,INCREMENTAL,Y,SCD2
hosts,workspace.raw.hosts,workspace.bronze.hosts,host_id,joined_at,INCREMENTAL,Y,SCD2
properties,workspace.raw.properties,workspace.bronze.properties,property_id,created_at,INCREMENTAL,Y,SCD1
bookings,workspace.raw.bookings,workspace.bronze.bookings,booking_id,updated_at,INCREMENTAL,Y,SCD1
payments,workspace.raw.payments,workspace.bronze.payments,payment_id,payment_date,INCREMENTAL,Y,SCD1
booking_updates,workspace.raw.booking_updates,workspace.bronze.booking_updates,booking_update_id,updated_at,INCREMENTAL,Y,SCD1


In [0]:
watermark_df = spark.table("workspace.metadata.watermark_tracker")

display(watermark_df)

table_name,last_watermark,current_watermark,last_run_status,last_run_time


In [0]:
def validate_schema(source_df, bronze_table):

    # First Load
    if not spark.catalog.tableExists(bronze_table):
        return True, [], [], []

    bronze_schema = spark.table(bronze_table).schema
    source_schema = source_df.schema

    bronze_cols = {f.name: str(f.dataType) for f in bronze_schema.fields}
    source_cols = {f.name: str(f.dataType) for f in source_schema.fields}

    new_columns = [c for c in source_cols if c not in bronze_cols]

    missing_columns = [c for c in bronze_cols if c not in source_cols]

    datatype_changes = []

    for c in set(source_cols).intersection(bronze_cols):
        if source_cols[c] != bronze_cols[c]:
            datatype_changes.append(
                (c, bronze_cols[c], source_cols[c])
            )

    is_valid = (
        len(missing_columns) == 0 and
        len(datatype_changes) == 0
    )

    return (
        is_valid,
        new_columns,
        missing_columns,
        datatype_changes
    )

In [0]:
import uuid

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
    DoubleType
)

In [0]:
from pyspark.sql.functions import col, lit, max
from datetime import datetime

for row in metadata_df.collect():

    table_name = row["table_name"]
    source_table = row["source_table"]
    target_table = row["target_table"]
    watermark_column = row["watermark_column"]
    load_type = row["load_type"]

    start_time = datetime.now()

    record_count = 0
    status = "SUCCESS"
    error_message = None

    try:

        print(f"\n========== Processing : {table_name} ==========")

        table_exists = spark.catalog.tableExists(target_table)

        # -------------------------
        # READ SOURCE
        # -------------------------

        if (not table_exists) or (load_type.upper() == "FULL"):

            print("First / Full Load")
            source_df = spark.table(source_table)

        else:

            wm = (
                spark.table("workspace.metadata.watermark_tracker")
                .filter(col("table_name") == table_name)
                .select("last_watermark")
                .collect()
            )

            if len(wm) == 0 or wm[0]["last_watermark"] is None:
                last_watermark = "1900-01-01 00:00:00"
            else:
                last_watermark = wm[0]["last_watermark"]

            print(f"Last Watermark : {last_watermark}")

            source_df = (
                spark.table(source_table)
                .filter(col(watermark_column) > lit(last_watermark))
            )
# -------------------------
# SCHEMA VALIDATION
# -------------------------
        is_valid, new_columns, missing_columns, datatype_changes = validate_schema(
            source_df,
            target_table
)

        if new_columns:
              print(f"New Columns Detected : {new_columns}")
              print("Schema Evolution Enabled")

        if missing_columns:
              raise Exception(f"Missing Columns : {missing_columns}")

        if datatype_changes:
              raise Exception(f"Datatype Changes : {datatype_changes}")

        record_count = source_df.count()

        print(f"Records Found : {record_count}")

        if record_count == 0:
            print("No New Records")

        else:

            write_mode = "overwrite" if not table_exists else "append"

            (
                source_df.write
                .format("delta")
                .option("mergeSchema", "true")
                .mode(write_mode)
                .saveAsTable(target_table)
            )

            print("Bronze Load Successful")

            # -------------------------
            # UPDATE WATERMARK
            # -------------------------

            new_watermark = (
                source_df
                .agg(max(col(watermark_column)).alias("wm"))
                .collect()[0]["wm"]
            )

            spark.sql(f"""
            MERGE INTO workspace.metadata.watermark_tracker t
            USING (
                SELECT
                    '{table_name}' AS table_name,
                    TIMESTAMP('{new_watermark}') AS last_watermark,
                    current_timestamp() AS last_run_time
            ) s
            ON t.table_name = s.table_name

            WHEN MATCHED THEN
              UPDATE SET
                t.last_watermark = s.last_watermark,
                t.last_run_time = s.last_run_time

            WHEN NOT MATCHED THEN
              INSERT (table_name,last_watermark,last_run_time)
              VALUES (s.table_name,s.last_watermark,s.last_run_time)
            """)

            print("Watermark Updated")

    except Exception as e:

        status = "FAILED"
        error_message = str(e)

        print(error_message)

    finally:

      end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_written", LongType(), True),
        StructField("rows_rejected", LongType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True)
    ])

    audit_data = [
        (
            str(uuid.uuid4()),
            table_name,
            "bronze",
            load_type,
            int(record_count),
            int(record_count),
            0,
            status,
            error_message if error_message else "",
            start_time,
            end_time,
            float(duration)
        )
    ]

    audit_df = spark.createDataFrame(
        audit_data,
        schema=audit_schema
    )

    audit_df.write.mode("append").saveAsTable(
        "workspace.metadata.audit_log"
    )

    print(f"Audit Logged : {table_name}")


========== Processing : users ==========
Last Watermark : 2025-07-30 23:05:18
Records Found : 0
No New Records
Audit Logged : users

========== Processing : hosts ==========
Last Watermark : 2025-07-30 00:00:00
Records Found : 0
No New Records
Audit Logged : hosts

========== Processing : properties ==========
Last Watermark : 2025-07-30 00:00:00
Records Found : 0
No New Records
Audit Logged : properties

========== Processing : bookings ==========
Last Watermark : 2025-07-31 23:59:59
Records Found : 0
No New Records
Audit Logged : bookings

========== Processing : payments ==========
Last Watermark : 2025-07-30 23:59:59
Records Found : 0
No New Records
Audit Logged : payments

========== Processing : booking_updates ==========
Last Watermark : 2025-07-31 23:59:59
Records Found : 0
No New Records
Audit Logged : booking_updates


In [0]:
%sql
SELECT * FROM workspace.metadata.watermark_tracker;

table_name,last_watermark,current_watermark,last_run_status,last_run_time
booking_updates,2025-07-31T23:59:59.000Z,null,null,2026-07-25T17:59:01.795Z
properties,2025-07-30T00:00:00.000Z,null,null,2026-07-25T17:58:34.838Z
bookings,2025-07-31T23:59:59.000Z,null,null,2026-07-25T17:58:49.201Z
payments,2025-07-30T23:59:59.000Z,null,null,2026-07-25T17:58:55.562Z
users,2025-07-30T23:05:18.000Z,null,null,2026-07-25T17:58:28.067Z
hosts,2025-07-30T00:00:00.000Z,null,null,2026-07-25T17:58:41.866Z


In [0]:
%sql
SELECT * FROM workspace.metadata.audit_log;

run_id,table_name,layer,load_type,rows_read,rows_written,rows_rejected,status,error_message,start_time,end_time,duration_seconds
73bb850a-d63c-4089-ac0d-21ad2c8a3f48,users,bronze,INCREMENTAL,124509,124509,0,SUCCESS,,2026-07-25T17:58:23.283Z,2026-07-25T17:58:30.114Z,6.831566
c9ac4cf9-4f87-4c5b-a4b0-292273ee3a48,properties,bronze,INCREMENTAL,18163,18163,0,SUCCESS,,2026-07-25T17:58:31.549Z,2026-07-25T17:58:36.999Z,5.450569
3a0a0230-4d56-4ca0-be38-86125cc56898,hosts,bronze,INCREMENTAL,19384,19384,0,SUCCESS,,2026-07-25T17:58:38.313Z,2026-07-25T17:58:44.264Z,5.951716
146425e5-2f84-447d-bb86-06cb648c658b,bookings,bronze,INCREMENTAL,72247,72247,0,SUCCESS,,2026-07-25T17:58:45.498Z,2026-07-25T17:58:51.341Z,5.842522
a2a6d045-1936-4d01-b7ae-25b7a5448a9c,payments,bronze,INCREMENTAL,49638,49638,0,SUCCESS,,2026-07-25T17:58:52.587Z,2026-07-25T17:58:57.652Z,5.065417
dc59a7e8-f25e-44b3-8581-c07040d245f9,booking_updates,bronze,INCREMENTAL,83068,83068,0,SUCCESS,,2026-07-25T17:58:58.897Z,2026-07-25T17:59:03.752Z,4.854619
22707cbe-e91d-461c-a89c-5579ac5f30c2,users,Silver,INCREMENTAL,124509,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:43:43.638Z,2026-07-28T10:43:50.082Z,6.443983
85ab1be7-188c-4ced-addf-85261cf948bf,hosts,Silver,INCREMENTAL,38768,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:43:51.953Z,2026-07-28T10:43:58.336Z,6.38358
8a220ac1-f797-4787-a5f1-20186cc6de7a,properties,Silver,INCREMENTAL,36326,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:43:59.643Z,2026-07-28T10:44:06.152Z,6.509818
5768b6f7-bbfd-4a8a-b555-a45b3f98d3b2,bookings,Silver,INCREMENTAL,144494,0,0,FAILED,[CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.,2026-07-28T10:44:07.659Z,2026-07-28T10:44:13.925Z,6.266121
